# 🔍 PHÂN TÍCH VÀ KHẮC PHỤC: Training Không Cải Thiện

## 📊 Nguyên Nhân Phổ Biến:

### 1️⃣ **Learning Rate Không Phù Hợp**
- **Quá cao**: Loss tăng giảm thất thường, không converge
- **Quá thấp**: Training chậm, dễ stuck ở local minima
- **Giải pháp**: 
  - Thử learning rate scheduler (ReduceLROnPlateau, CosineAnnealing)
  - Test các giá trị: 1e-4, 5e-5, 1e-5
  - Dùng Learning Rate Finder

### 2️⃣ **Data Quality Issues**
- **Data statistics không đúng**: Mel mean/std tính sai
- **Data imbalance**: Dữ liệu không đồng đều
- **Noise trong data**: Audio quality thấp
- **Giải pháp**:
  - Kiểm tra lại mel_mean, mel_std
  - Kiểm tra quality của audio files
  - Chuẩn hóa dữ liệu tốt hơn

### 3️⃣ **Model Capacity Issues**
- **Underfitting**: Model quá nhỏ, không đủ capacity
- **Overfitting**: Model quá lớn, học thuộc training data
- **Giải pháp**:
  - Monitor train_loss vs val_loss
  - Thêm/giảm regularization (dropout, weight_decay)
  - Điều chỉnh model size

### 4️⃣ **Gradient Problems**
- **Vanishing gradients**: Gradients quá nhỏ
- **Exploding gradients**: Gradients quá lớn
- **Giải pháp**:
  - Kiểm tra gradient norms trong logs
  - Điều chỉnh gradient_clip_val
  - Thử mixed precision training

### 5️⃣ **Batch Size & Accumulation**
- **Batch quá nhỏ**: Gradients không stable
- **Batch quá lớn**: Generalization kém
- **Giải pháp**:
  - Effective batch = batch_size × accumulate_grad_batches × devices
  - Thử tăng/giảm accumulation
  - Điều chỉnh learning rate theo batch size

### 6️⃣ **LLM Frozen Issue**
- **PhoBERT frozen**: Prosody features không adapt
- **Giải pháp**:
  - Thử `finetune_llm: True` (nếu đủ VRAM)
  - Hoặc train prosody projection layer lâu hơn

### 7️⃣ **Validation Check Frequency**
- **val_check_interval quá cao**: Không catch được best model
- **Giải pháp**: Giảm từ 0.5 → 0.25 hoặc 1.0

---

## 🛠️ Khuyến Nghị Cụ Thể Cho Project Này:

### ✅ Thử các cải tiến sau:

1. **Learning Rate Scheduler**:
   ```python
   from lightning.pytorch.callbacks import LearningRateMonitor
   from torch.optim.lr_scheduler import ReduceLROnPlateau
   
   # Trong optimizer_kwargs
   "lr_scheduler": {
       "scheduler": ReduceLROnPlateau,
       "monitor": "loss/val_epoch",
       "factor": 0.5,
       "patience": 5,
   }
   ```

2. **Warmup Steps**:
   - Thêm warmup cho 1000-2000 steps đầu
   - Tăng dần learning rate từ 0 → target_lr

3. **Data Augmentation**:
   - SpecAugment cho mel-spectrogram
   - Time stretching, pitch shifting

4. **Monitor Thêm Metrics**:
   - Train/Val loss gap (overfitting indicator)
   - Gradient norms
   - Learning rate actual value

5. **Checkpoint Strategy**:
   - Lưu checkpoint theo interval (mỗi N epochs)
   - Không chỉ dựa vào val_loss

6. **Resume from Checkpoint**:
   - Nếu train nhiều lần, resume thay vì restart
   - Accumulate knowledge từ các lần train trước

## 🔬 DIAGNOSTICS: Kiểm Tra Nguyên Nhân

In [ ]:
# ============================================================================
# 🔬 DIAGNOSTIC TOOLS: Phân tích training logs
# ============================================================================
import os
import glob
import json
from pathlib import Path

def analyze_training_logs(log_dir="outputs/matcha_prosody_fixed/logs"):
    """Phân tích TensorBoard logs để tìm nguyên nhân"""
    
    print("="*80)
    print("🔍 PHÂN TÍCH TRAINING LOGS")
    print("="*80)
    
    # 1. Kiểm tra logs có tồn tại không
    if not os.path.exists(log_dir):
        print(f"❌ Không tìm thấy log directory: {log_dir}")
        return
    
    # 2. Tìm tất cả version folders
    version_dirs = sorted(glob.glob(os.path.join(log_dir, "version_*")))
    
    if not version_dirs:
        print(f"❌ Không tìm thấy training versions trong {log_dir}")
        return
    
    print(f"✅ Tìm thấy {len(version_dirs)} training sessions:")
    for vdir in version_dirs:
        print(f"  • {os.path.basename(vdir)}")
    
    # 3. Phân tích version mới nhất
    latest_version = version_dirs[-1]
    print(f"\n📊 Phân tích session mới nhất: {os.path.basename(latest_version)}")
    
    # 4. Đọc hparams
    hparams_file = os.path.join(latest_version, "hparams.yaml")
    if os.path.exists(hparams_file):
        print(f"\n✅ Hyperparameters:")
        with open(hparams_file, 'r') as f:
            for i, line in enumerate(f):
                if i < 15:  # Show first 15 lines
                    print(f"  {line.rstrip()}")
        print("  ...")
    
    # 5. Kiểm tra TensorBoard events
    event_files = glob.glob(os.path.join(latest_version, "events.out.tfevents.*"))
    if event_files:
        print(f"\n✅ TensorBoard events: {len(event_files)} file(s)")
        print(f"  📁 Xem chi tiết bằng TensorBoard:")
        print(f"     %tensorboard --logdir {log_dir}")
    else:
        print(f"\n⚠️  Không tìm thấy TensorBoard events")
    
    # 6. Suggest next steps
    print("\n" + "="*80)
    print("💡 KHUYẾN NGHỊ:")
    print("="*80)
    print("1. Mở TensorBoard để xem train/val loss curves")
    print("2. Kiểm tra gradient norms có bất thường không")
    print("3. So sánh learning rate actual vs expected")
    print("4. Xem distribution của weights/activations")
    print("="*80)

def check_data_quality(filelist_path, n_samples=10):
    """Kiểm tra chất lượng dữ liệu"""
    
    print("\n" + "="*80)
    print("🔍 KIỂM TRA CHẤT LƯỢNG DỮ LIỆU")
    print("="*80)
    
    if not os.path.exists(filelist_path):
        print(f"❌ Không tìm thấy filelist: {filelist_path}")
        return
    
    with open(filelist_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    print(f"✅ Tổng số samples: {len(lines)}")
    
    # Check format
    print(f"\n📋 Kiểm tra format (sample đầu tiên):")
    if lines:
        parts = lines[0].strip().split('|')
        print(f"  Số cột: {len(parts)}")
        if len(parts) >= 3:
            print(f"  Audio path: {parts[0][:50]}...")
            print(f"  Raw text: {parts[1][:50]}...")
            print(f"  IPA text: {parts[2][:50]}...")
    
    # Check audio files existence
    print(f"\n🎵 Kiểm tra {n_samples} audio files ngẫu nhiên:")
    import random
    sample_lines = random.sample(lines, min(n_samples, len(lines)))
    
    missing_count = 0
    for line in sample_lines:
        audio_path = line.strip().split('|')[0]
        if not os.path.exists(audio_path):
            print(f"  ❌ Thiếu: {audio_path}")
            missing_count += 1
    
    if missing_count == 0:
        print(f"  ✅ Tất cả {n_samples} files đều tồn tại!")
    else:
        print(f"  ⚠️  Thiếu {missing_count}/{n_samples} files")
    
    print("="*80)

def suggest_improvements():
    """Gợi ý các cải tiến cụ thể"""
    
    print("\n" + "="*80)
    print("🚀 GỢI Ý CẢI TIẾN TRAINING")
    print("="*80)
    
    improvements = [
        {
            "title": "1. Thêm Learning Rate Scheduler",
            "why": "Tự động giảm LR khi val_loss plateau",
            "how": "Thêm ReduceLROnPlateau vào trainer",
            "priority": "HIGH"
        },
        {
            "title": "2. Tăng Gradient Accumulation",
            "why": "Effective batch size lớn hơn → training stable hơn",
            "how": "Tăng accumulate_grad_batches: 4 → 8",
            "priority": "MEDIUM"
        },
        {
            "title": "3. Warmup Learning Rate",
            "why": "Tránh loss spike ở đầu training",
            "how": "Thêm warmup_steps = 1000",
            "priority": "HIGH"
        },
        {
            "title": "4. Fine-tune LLM",
            "why": "Prosody features adapt tốt hơn với dữ liệu",
            "how": "Đặt finetune_llm=True (cần thêm VRAM)",
            "priority": "MEDIUM"
        },
        {
            "title": "5. Data Augmentation",
            "why": "Tăng diversity, giảm overfitting",
            "how": "Thêm SpecAugment, time stretching",
            "priority": "LOW"
        },
        {
            "title": "6. Thử Learning Rate khác",
            "why": "LR hiện tại có thể không optimal",
            "how": "Test: 1e-4, 2e-5, 1e-5",
            "priority": "HIGH"
        },
        {
            "title": "7. Monitor Thêm Metrics",
            "why": "Hiểu rõ hơn training dynamics",
            "how": "Log gradient norms, weight distributions",
            "priority": "MEDIUM"
        },
    ]
    
    for imp in improvements:
        print(f"\n{imp['title']} [{imp['priority']}]")
        print(f"  Tại sao: {imp['why']}")
        print(f"  Cách làm: {imp['how']}")
    
    print("\n" + "="*80)

# Run diagnostics
print("🔬 CHẠY DIAGNOSTIC CHECKS...")
print("\nNOTE: Update đường dẫn cho đúng với môi trường của bạn\n")

# Uncomment để chạy:
# analyze_training_logs("outputs/matcha_prosody_fixed/logs")
# check_data_quality("data/train_filelist.txt", n_samples=10)
suggest_improvements()

print("\n✅ Diagnostic hoàn tất! Xem các gợi ý phía trên.")

## 🔧 IMPROVED TRAINING SCRIPT

Script training được cải tiến với:
- ✅ Learning Rate Scheduler (ReduceLROnPlateau)
- ✅ Warmup steps
- ✅ Tăng monitoring metrics
- ✅ Flexible checkpoint strategy
- ✅ Better gradient handling

In [ ]:
# ============================================================================
# ✅ KIỂM TRA & ĐỊNH NGHĨA BIẾN CẦN THIẾT
# ============================================================================
import os

print("="*80)
print("🔍 KIỂM TRA CÁC BIẾN CẦN THIẾT")
print("="*80)

# Kiểm tra và định nghĩa các biến nếu chưa tồn tại
required_vars = {}

# 1. TRAIN_LIST_OUTPUT
try:
    required_vars['TRAIN_LIST_OUTPUT'] = TRAIN_LIST_OUTPUT
    print(f"✅ TRAIN_LIST_OUTPUT = {TRAIN_LIST_OUTPUT}")
except NameError:
    print("⚠️  TRAIN_LIST_OUTPUT chưa được định nghĩa")
    print("   Dùng giá trị mặc định cho Kaggle")
    TRAIN_LIST_OUTPUT = "/kaggle/working/fixed_train.txt"
    required_vars['TRAIN_LIST_OUTPUT'] = TRAIN_LIST_OUTPUT
    print(f"   → {TRAIN_LIST_OUTPUT}")

# 2. VAL_LIST_OUTPUT
try:
    required_vars['VAL_LIST_OUTPUT'] = VAL_LIST_OUTPUT
    print(f"✅ VAL_LIST_OUTPUT = {VAL_LIST_OUTPUT}")
except NameError:
    print("⚠️  VAL_LIST_OUTPUT chưa được định nghĩa")
    VAL_LIST_OUTPUT = "/kaggle/working/fixed_val.txt"
    required_vars['VAL_LIST_OUTPUT'] = VAL_LIST_OUTPUT
    print(f"   → {VAL_LIST_OUTPUT}")

# 3. AUDIO_DIR
try:
    required_vars['AUDIO_DIR'] = AUDIO_DIR
    print(f"✅ AUDIO_DIR = {AUDIO_DIR}")
except NameError:
    print("⚠️  AUDIO_DIR chưa được định nghĩa")
    AUDIO_DIR = "/kaggle/input/data-audio-ipa/kaggle/working/data/subs_add_con"
    required_vars['AUDIO_DIR'] = AUDIO_DIR
    print(f"   → {AUDIO_DIR}")

# 4. CALCULATED_MEAN
try:
    required_vars['CALCULATED_MEAN'] = CALCULATED_MEAN
    print(f"✅ CALCULATED_MEAN = {CALCULATED_MEAN:.4f}")
except NameError:
    print("⚠️  CALCULATED_MEAN chưa được tính")
    CALCULATED_MEAN = -4.5129  # Default value
    required_vars['CALCULATED_MEAN'] = CALCULATED_MEAN
    print(f"   → Dùng giá trị mặc định: {CALCULATED_MEAN:.4f}")

# 5. CALCULATED_STD
try:
    required_vars['CALCULATED_STD'] = CALCULATED_STD
    print(f"✅ CALCULATED_STD = {CALCULATED_STD:.4f}")
except NameError:
    print("⚠️  CALCULATED_STD chưa được tính")
    CALCULATED_STD = 2.3453  # Default value
    required_vars['CALCULATED_STD'] = CALCULATED_STD
    print(f"   → Dùng giá trị mặc định: {CALCULATED_STD:.4f}")

print("\n" + "="*80)
print("📋 TÓM TẮT:")
print("="*80)
all_defined = True
for var_name, var_value in required_vars.items():
    print(f"  {var_name}: {var_value}")
    
print("\n💡 LƯU Ý:")
print("  • Nếu dùng giá trị mặc định, hãy chạy lại các cell tính stats (cell 8, 9)")
print("  • Nếu paths không đúng, hãy cập nhật trong các cell định nghĩa")
print("="*80)

In [ ]:
# ============================================================================
# 🚀 IMPROVED TRAINING SCRIPT với Learning Rate Scheduler & Warmup
# ============================================================================

improved_script = f"""
import sys
import os
import torch
from argparse import Namespace

# ============================================================================
# IMPROVED TRAINING SCRIPT - MATCHA-TTS WITH LLM PROSODY
# ============================================================================

from lightning import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.strategies import DDPStrategy
from matcha.models.matcha_tts import MatchaTTS
from matcha.data.text_mel_datamodule import TextMelDataModule
from matcha.text.symbols import symbols
import torch.serialization

# Fix pickle
torch.serialization.add_safe_globals([Namespace])

# ============================================================================
# CONFIG TRAINING - IMPROVED VERSION
# ============================================================================
CONFIG = {{
    "train_filelist": "{TRAIN_LIST_OUTPUT}",
    "val_filelist": "{VAL_LIST_OUTPUT}",
    "output_dir": "outputs/matcha_prosody_improved",
    
    # Model
    "n_spks": 1,
    "spk_emb_dim": 64,
    "n_feats": 80,
    
    # Prosody
    "llm_model_name": "vinai/phobert-base",
    "prosody_dim": 192,
    "use_token_level_prosody": True,
    "finetune_llm": False,  # Set True nếu có đủ VRAM (cần thêm ~3GB)
    
    # Training - IMPROVED
    "batch_size": 4,
    "learning_rate": 1e-4,  # ✅ Tăng từ 5e-5 (thử aggressive hơn)
    "warmup_steps": 1000,   # ✅ THÊM warmup
    "max_epochs": 200,
    "num_workers": 2,
    "accumulate_grad_batches": 8,  # ✅ Tăng từ 4 → 8 (effective batch = 64)
    
    # Scheduler
    "use_scheduler": True,      # ✅ THÊM scheduler
    "scheduler_patience": 10,    # Giảm LR sau 10 epochs không cải thiện
    "scheduler_factor": 0.5,     # Giảm LR xuống 50%
    "min_lr": 1e-6,             # Learning rate tối thiểu
    
    # Hardware
    "accelerator": "gpu",
    "devices": 2,
    "precision": "16-mixed",
    
    # Monitoring
    "val_check_interval": 0.5,   # Check validation 2 lần/epoch
    "log_every_n_steps": 25,     # ✅ Giảm từ 50 (log nhiều hơn)
    
    # Paths
    "audio_root": "{AUDIO_DIR}",
    "resume_from_checkpoint": None,  # Set path nếu muốn resume
}}

CONFIG["n_vocab"] = len(symbols) + 1

# Data stats
DATA_STATS = {{
    "mel_mean": {CALCULATED_MEAN},
    "mel_std": {CALCULATED_STD},
}}

# ============================================================================
# ENCODER CONFIG - OPTIMIZED
# ============================================================================
ENCODER_CONFIG = {{
    "encoder_type": "RoPE Encoder",
    "encoder_params": {{
        "n_feats": CONFIG["n_feats"],
        "n_channels": 512,
        "filter_channels": 1536,
        "filter_channels_dp": 384,
        "n_heads": 8,
        "n_layers": 8,
        "kernel_size": 3,
        "p_dropout": 0.1,
        "spk_emb_dim": CONFIG["spk_emb_dim"],
        "n_spks": CONFIG["n_spks"],
        "prenet": True,
    }},
    "duration_predictor_params": {{
        "filter_channels_dp": 256,
        "kernel_size": 3,
        "p_dropout": 0.1,
    }},
}}

# ============================================================================
# DECODER CONFIG - OPTIMIZED
# ============================================================================
DECODER_CONFIG = {{
    "channels": [256, 256],
    "dropout": 0.2,
    "attention_head_dim": 64,
    "n_blocks": 4,
    "num_mid_blocks": 4,
    "num_heads": 8,
    "act_fn": "gelu",
}}

CFM_CONFIG = {{
    "sigma_min": 1e-4,
    "solver": "euler",
    "t_scheduler": "cosine",
}}

print("="*80)
print("🍵 MATCHA-TTS IMPROVED TRAINING")
print("="*80)
print(f"✅ Effective Batch: {{CONFIG['batch_size']}} × {{CONFIG['accumulate_grad_batches']}} × {{CONFIG['devices']}} = {{CONFIG['batch_size'] * CONFIG['accumulate_grad_batches'] * CONFIG['devices']}}")
print(f"✅ Initial LR: {{CONFIG['learning_rate']}}")
print(f"✅ Warmup Steps: {{CONFIG['warmup_steps']}}")
print(f"✅ LR Scheduler: {{CONFIG['use_scheduler']}}")
print(f"✅ Fine-tune LLM: {{CONFIG['finetune_llm']}}")
print("="*80)

def create_model():
    encoder = Namespace(
        encoder_type=ENCODER_CONFIG["encoder_type"],
        encoder_params=Namespace(**ENCODER_CONFIG["encoder_params"]),
        duration_predictor_params=Namespace(**ENCODER_CONFIG["duration_predictor_params"]),
    )
    
    decoder = DECODER_CONFIG.copy()
    cfm = Namespace(**CFM_CONFIG)
    
    model = MatchaTTS(
        n_vocab=CONFIG["n_vocab"],
        n_spks=CONFIG["n_spks"],
        spk_emb_dim=CONFIG["spk_emb_dim"],
        n_feats=CONFIG["n_feats"],
        encoder=encoder,
        decoder=decoder,
        cfm=cfm,
        data_statistics=DATA_STATS,
        out_size=None,
        optimizer=torch.optim.AdamW,
        optimizer_kwargs={{
            "lr": CONFIG["learning_rate"], 
            "weight_decay": 1e-6,
            "betas": (0.9, 0.98),  # ✅ Thêm betas cho stable training
            "eps": 1e-9,
        }},
        llm_model_name=CONFIG["llm_model_name"],
        prosody_dim=CONFIG["prosody_dim"],
        use_token_level_prosody=CONFIG["use_token_level_prosody"],
        finetune_llm=CONFIG["finetune_llm"],
    )
    
    return model

def create_datamodule():
    datamodule = TextMelDataModule(
        name="matcha_vi",
        train_filelist_path=CONFIG["train_filelist"],
        valid_filelist_path=CONFIG["val_filelist"],
        batch_size=CONFIG["batch_size"],
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
        cleaners=["basic_cleaners_phothong"],
        add_blank=True,
        n_spks=CONFIG["n_spks"],
        n_fft=1024,
        n_feats=CONFIG["n_feats"],
        sample_rate=22050,
        hop_length=256,
        win_length=1024,
        f_min=0,
        f_max=8000,
        data_statistics=DATA_STATS,
        seed=1234,
        load_durations=False,
        audio_root=CONFIG["audio_root"],
    )
    return datamodule

# ============================================================================
# 🆕 CUSTOM CALLBACK: Warmup + ReduceLROnPlateau
# ============================================================================
from lightning.pytorch.callbacks import Callback

class WarmupLR(Callback):
    def __init__(self, warmup_steps, target_lr):
        super().__init__()
        self.warmup_steps = warmup_steps
        self.target_lr = target_lr
        self.current_step = 0
        
    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        if self.current_step < self.warmup_steps:
            lr_scale = min(1.0, float(self.current_step + 1) / float(self.warmup_steps))
            for pg in trainer.optimizers[0].param_groups:
                pg['lr'] = lr_scale * self.target_lr
        self.current_step += 1

class ReduceLROnPlateauCallback(Callback):
    def __init__(self, monitor='loss/val_epoch', patience=10, factor=0.5, min_lr=1e-6):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.factor = factor
        self.min_lr = min_lr
        self.best_score = None
        self.wait = 0
        
    def on_validation_end(self, trainer, pl_module):
        current = trainer.callback_metrics.get(self.monitor)
        if current is None:
            return
            
        if self.best_score is None:
            self.best_score = current
        elif current < self.best_score:
            self.best_score = current
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                for pg in trainer.optimizers[0].param_groups:
                    old_lr = pg['lr']
                    new_lr = max(old_lr * self.factor, self.min_lr)
                    pg['lr'] = new_lr
                    print(f"\\n🔽 Reducing learning rate: {{old_lr:.2e}} → {{new_lr:.2e}}")
                self.wait = 0

def train():
    # Init
    model = create_model()
    datamodule = create_datamodule()

    # Callbacks
    checkpoint_cb = ModelCheckpoint(
        dirpath=CONFIG["output_dir"] + "/checkpoints",
        filename="matcha-{{epoch:03d}}-{{step:06d}}-{{loss/val_epoch:.4f}}",  # ✅ Thêm step vào filename
        monitor="loss/val_epoch",
        mode="min",
        save_top_k=5,  # ✅ Tăng từ 3 → 5
        save_last=True,
        every_n_epochs=5,  # ✅ THÊM: Lưu mỗi 5 epochs
    )
    
    lr_monitor = LearningRateMonitor(logging_interval="step")
    
    # ✅ THÊM custom callbacks
    warmup_cb = WarmupLR(
        warmup_steps=CONFIG["warmup_steps"],
        target_lr=CONFIG["learning_rate"]
    )
    
    callbacks_list = [checkpoint_cb, lr_monitor, warmup_cb]
    
    if CONFIG["use_scheduler"]:
        scheduler_cb = ReduceLROnPlateauCallback(
            monitor="loss/val_epoch",
            patience=CONFIG["scheduler_patience"],
            factor=CONFIG["scheduler_factor"],
            min_lr=CONFIG["min_lr"]
        )
        callbacks_list.append(scheduler_cb)

    # Trainer with DDP
    strategy = DDPStrategy(find_unused_parameters=True)
    
    trainer = Trainer(
        accelerator=CONFIG["accelerator"],
        devices=CONFIG["devices"],
        strategy=strategy,
        max_epochs=CONFIG["max_epochs"],
        callbacks=callbacks_list,
        logger=TensorBoardLogger(CONFIG["output_dir"], name="logs"),
        precision=CONFIG["precision"],
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",  # ✅ THÊM: Chỉ định algorithm
        accumulate_grad_batches=CONFIG["accumulate_grad_batches"],
        log_every_n_steps=CONFIG["log_every_n_steps"],
        val_check_interval=CONFIG["val_check_interval"],
        # ✅ THÊM: Detect anomaly nếu cần debug
        # detect_anomaly=True,  # Uncomment nếu có NaN/Inf
    )

    # Train
    print("\\n🚀 Starting improved training...")
    trainer.fit(
        model, 
        datamodule=datamodule,
        ckpt_path=CONFIG["resume_from_checkpoint"]
    )
    
    print("\\n" + "="*80)
    print("✅ TRAINING COMPLETED!")
    print(f"📁 Checkpoints saved to: {{CONFIG['output_dir']}}/checkpoints")
    print(f"📊 TensorBoard logs: {{CONFIG['output_dir']}}/logs")
    print("="*80)

if __name__ == "__main__":
    train()
"""

# Save improved script
workspace_root = os.getcwd()
improved_script_path = os.path.join(workspace_root, "train_matcha_improved.py")

with open(improved_script_path, "w") as f:
    f.write(improved_script)

print("="*80)
print("✅ IMPROVED TRAINING SCRIPT")
print("="*80)
print(f"📝 Đã lưu script cải tiến tại: {improved_script_path}")
print(f"\n🔧 Các cải tiến chính:")
print("  1. ✅ Learning Rate Warmup (1000 steps)")
print("  2. ✅ ReduceLROnPlateau scheduler")
print("  3. ✅ Tăng effective batch size (64)")
print("  4. ✅ Better optimizer config (betas, eps)")
print("  5. ✅ Checkpoint mỗi 5 epochs")
print("  6. ✅ Tăng monitoring frequency")
print("\n💡 So sánh với script cũ:")
print(f"  Old: train_matcha_final.py")
print(f"  New: train_matcha_improved.py")
print("\n🚀 Chạy bằng lệnh:")
print(f"  !python {improved_script_path}")
print("="*80)

## 📊 Quick Comparison: Training Configs

In [ ]:
# ============================================================================
# 📊 SO SÁNH CONFIG: Original vs Improved
# ============================================================================
import pandas as pd

comparison_data = {
    "Parameter": [
        "Learning Rate",
        "Warmup Steps",
        "LR Scheduler",
        "Effective Batch Size",
        "Accumulate Grad",
        "Gradient Clip",
        "Checkpoint Strategy",
        "Log Frequency",
        "Val Check Interval",
        "Optimizer Betas",
        "Save Top K",
    ],
    "Original Script": [
        "5e-5",
        "❌ None",
        "❌ None",
        "32 (4×4×2)",
        "4",
        "1.0 (default)",
        "Top 3 by val_loss",
        "Every 50 steps",
        "0.5 (2x/epoch)",
        "(0.9, 0.999) default",
        "3",
    ],
    "Improved Script": [
        "1e-4 (2x higher)",
        "✅ 1000 steps",
        "✅ ReduceLROnPlateau",
        "64 (4×8×2)",
        "8 (2x higher)",
        "1.0 norm",
        "Top 5 + every 5 epochs",
        "Every 25 steps (2x)",
        "0.5 (same)",
        "(0.9, 0.98) tuned",
        "5",
    ],
    "Impact": [
        "Faster convergence",
        "Stable start",
        "Auto adapt LR",
        "More stable gradients",
        "Better gradient estimate",
        "Prevent explosion",
        "More checkpoints",
        "Better monitoring",
        "Same validation freq",
        "Better for TTS",
        "More backup",
    ]
}

df = pd.DataFrame(comparison_data)

print("="*100)
print("📊 TRAINING CONFIG COMPARISON")
print("="*100)
print(df.to_string(index=False))
print("="*100)

print("\n💡 KEY IMPROVEMENTS:")
print("  1. 🔥 Warmup prevents loss spikes at start")
print("  2. 📉 LR Scheduler auto-reduces when plateaued")
print("  3. 💪 2x effective batch = more stable training")
print("  4. 📊 Better monitoring (2x logging frequency)")
print("  5. 💾 More checkpoints = safety net")

print("\n⚠️  TRADEOFFS:")
print("  • Effective batch 64 → cần thêm ~15% VRAM")
print("  • Warmup 1000 steps → first epoch chậm hơn ~10%")
print("  • More checkpoints → cần thêm disk space")

print("\n🎯 WHEN TO USE WHICH:")
print("  Original: Khi VRAM hạn chế (<12GB)")
print("  Improved: Khi muốn training quality cao hơn (>=14GB)")
print("="*100)

## 🔍 Debug & Monitor Guide

In [ ]:
# ============================================================================
# 🔍 HƯỚNG DẪN DEBUG & MONITOR TRAINING
# ============================================================================

debug_guide = """
╔════════════════════════════════════════════════════════════════════════════╗
║                    🔍 DEBUG & MONITOR CHECKLIST                            ║
╚════════════════════════════════════════════════════════════════════════════╝

📊 1. MONITORING QUA TENSORBOARD
────────────────────────────────────────────────────────────────────────────
   Mở TensorBoard:
   >>> %tensorboard --logdir outputs/matcha_prosody_improved/logs
   
   Xem các metrics sau:
   
   ✅ SCALARS Tab:
      • loss/train_epoch vs loss/val_epoch
        → Train giảm mà Val không giảm = OVERFITTING
        → Cả 2 đều cao và không giảm = UNDERFITTING / LR sai
        → Cả 2 giảm đều = GOOD!
      
      • loss/train_step (realtime)
        → Smooth curve = training stable
        → Spiky/noisy = batch size quá nhỏ hoặc LR quá cao
      
      • lr-AdamW (learning rate actual)
        → Kiểm tra warmup có hoạt động không
        → Xem scheduler có reduce LR không
   
   ✅ DISTRIBUTIONS Tab:
      • Xem weight distributions
        → Nếu tất cả weights → 0 = dying neurons
        → Nếu explode = gradient explosion
   
   ✅ HISTOGRAMS Tab:
      • Gradient norms
        → Gradient < 1e-8 = vanishing
        → Gradient > 100 = exploding

────────────────────────────────────────────────────────────────────────────
📈 2. PHÂN TÍCH LOSS CURVES
────────────────────────────────────────────────────────────────────────────
   
   Tình huống A: Train Loss giảm, Val Loss plateau
   ────────────────────────────────────────────────────
   Nguyên nhân: OVERFITTING
   Giải pháp:
     • Tăng dropout: 0.1 → 0.2
     • Tăng weight_decay: 1e-6 → 1e-5
     • Thêm data augmentation
     • Giảm model size
   
   Tình huống B: Cả Train & Val Loss đều plateau cao
   ────────────────────────────────────────────────────
   Nguyên nhân: UNDERFITTING hoặc LR không phù hợp
   Giải pháp:
     • Tăng learning rate: 1e-4 → 2e-4
     • Tăng model capacity (n_layers, n_channels)
     • Train lâu hơn
     • Kiểm tra data quality
   
   Tình huống C: Loss tăng giảm thất thường
   ────────────────────────────────────────────────────
   Nguyên nhân: LR quá cao hoặc gradient exploding
   Giải pháp:
     • Giảm learning rate: 1e-4 → 5e-5
     • Giảm gradient_clip_val: 1.0 → 0.5
     • Kiểm tra batch có NaN không
   
   Tình huống D: Loss giảm rất chậm
   ────────────────────────────────────────────────────
   Nguyên nhân: LR quá thấp
   Giải pháp:
     • Tăng learning rate
     • Giảm warmup_steps
     • Kiểm tra optimizer config

────────────────────────────────────────────────────────────────────────────
🛠️ 3. RUNTIME DEBUGGING
────────────────────────────────────────────────────────────────────────────
   
   Nếu training bị stuck hoặc lỗi:
   
   A. Check GPU Memory:
      >>> !nvidia-smi
      → Nếu OOM: giảm batch_size hoặc accumulate_grad_batches
   
   B. Check Data Loading:
      >>> # Thêm vào script
      >>> datamodule.setup()
      >>> batch = next(iter(datamodule.train_dataloader()))
      >>> print(batch.keys(), batch['x'].shape)
   
   C. Check Model Forward:
      >>> # Test một batch
      >>> model = create_model()
      >>> output = model.forward(batch)
      >>> print(output.keys())
   
   D. Enable Anomaly Detection:
      >>> # Trong Trainer, thêm:
      >>> detect_anomaly=True
      → Sẽ chậm hơn nhưng catch được NaN/Inf

────────────────────────────────────────────────────────────────────────────
📋 4. CHECKPOINT DEBUGGING
────────────────────────────────────────────────────────────────────────────
   
   Kiểm tra checkpoint:
   
   >>> import torch
   >>> ckpt = torch.load('path/to/checkpoint.ckpt', map_location='cpu')
   >>> print(ckpt.keys())
   >>> print(f"Epoch: {ckpt['epoch']}")
   >>> print(f"Global step: {ckpt['global_step']}")
   >>> print(f"Best val loss: {ckpt.get('checkpoint_callback_best_model_score')}")
   
   Resume từ checkpoint:
   
   >>> CONFIG["resume_from_checkpoint"] = "outputs/.../checkpoints/last.ckpt"
   >>> # Rerun training script

────────────────────────────────────────────────────────────────────────────
🎯 5. EXPERIMENTATION CHECKLIST
────────────────────────────────────────────────────────────────────────────
   
   Thử các thí nghiệm sau (1 lần 1 thay đổi):
   
   ✓ Learning Rate Sweep:
     • 2e-4, 1e-4, 5e-5, 2e-5, 1e-5
   
   ✓ Batch Size Sweep:
     • Effective batch: 32, 64, 128
   
   ✓ Model Size:
     • n_channels: 384, 512, 768
     • n_layers: 6, 8, 12
   
   ✓ Dropout:
     • p_dropout: 0.05, 0.1, 0.2
   
   ✓ Fine-tune LLM:
     • finetune_llm: True vs False
   
   ✓ Scheduler Patience:
     • patience: 5, 10, 15

────────────────────────────────────────────────────────────────────────────
💡 6. BEST PRACTICES
────────────────────────────────────────────────────────────────────────────
   
   1. Luôn monitor BOTH train & val loss
   2. Lưu logs của mỗi experiment vào folder riêng
   3. Document thay đổi (ghi chú vào TensorBoard hoặc file)
   4. So sánh các runs trong TensorBoard (chọn nhiều runs)
   5. Patience: Đợi ít nhất 20-30 epochs trước khi kết luận
   6. Resume training thay vì restart từ đầu
   7. Backup checkpoints tốt nhất

╚════════════════════════════════════════════════════════════════════════════╝
"""

print(debug_guide)

# Quick commands
print("\n" + "="*80)
print("🚀 QUICK COMMANDS")
print("="*80)
print("# 1. Xem TensorBoard")
print("%tensorboard --logdir outputs/matcha_prosody_improved/logs")
print("\n# 2. Check GPU")
print("!nvidia-smi")
print("\n# 3. List checkpoints")
print("!ls -lh outputs/matcha_prosody_improved/checkpoints/")
print("\n# 4. Find best checkpoint")
print("!ls outputs/matcha_prosody_improved/checkpoints/*.ckpt | sort -V")
print("="*80)

---

## 📝 TÓM TẮT: Khắc Phục Training Không Cải Thiện

### ✅ Đã Thực Hiện:

1. **Phân tích nguyên nhân** - 7 nguyên nhân chính được xác định
2. **Tạo script cải tiến** - `train_matcha_improved.py` với:
   - Learning Rate Warmup
   - ReduceLROnPlateau Scheduler
   - Tăng effective batch size
   - Better monitoring
3. **Diagnostic tools** - Công cụ phân tích logs và data quality
4. **Debug guide** - Hướng dẫn chi tiết monitor và troubleshoot

### 🎯 Các Bước Tiếp Theo:

1. **Chạy script cải tiến**: `!python train_matcha_improved.py`
2. **Monitor qua TensorBoard**: Theo dõi train/val loss curves
3. **Thử nghiệm learning rates**: Test 1e-4, 5e-5, 2e-5
4. **Kiểm tra data quality**: Đảm bảo mel_stats chính xác
5. **Resume từ checkpoint**: Nếu training gián đoạn

### 💡 Lưu Ý Quan Trọng:

- **Patience**: Đợi ít nhất 30-50 epochs trước khi kết luận
- **Comparison**: So sánh original vs improved script
- **Documentation**: Ghi chú mọi thay đổi và kết quả
- **Backup**: Lưu checkpoints tốt nhất

## 🚀 Hướng Dẫn Sử Dụng

### Để khắc phục vấn đề training không cải thiện:

**Bước 1: Chạy cell kiểm tra biến** (ngay phía trên)
- Đảm bảo tất cả biến `TRAIN_LIST_OUTPUT`, `VAL_LIST_OUTPUT`, `AUDIO_DIR`, `CALCULATED_MEAN`, `CALCULATED_STD` đã được định nghĩa
- Nếu chưa có, cell sẽ tự động dùng giá trị mặc định

**Bước 2: Chạy cell tạo improved script** (cell kế tiếp)
- Tạo file `train_matcha_improved.py` với các cải tiến

**Bước 3: Training**
```python
!python train_matcha_improved.py
```

**Bước 4: Monitor**
```python
%tensorboard --logdir outputs/matcha_prosody_improved/logs
```

**Nếu vẫn không cải thiện:**
1. Chạy cell **Diagnostic Tools** để phân tích logs
2. Thử nghiệm learning rates khác nhau (1e-4, 5e-5, 2e-5)
3. Kiểm tra data quality
4. Xem **Debug Guide** để troubleshoot chi tiết

# MATCHA-TTS Training Complete Pipeline for Kaggle
## Setup môi trường và training Matcha-TTS với Prosody Analysis (PhoBERT)

Notebook này tổng hợp toàn bộ code từ project để train trên Kaggle

In [ ]:
# 1. Clone repo từ GitHub & Checkout branch
!git clone https://github.com/Neyly112/IPIATTS.git
%cd IPIATTS
!git checkout ket_hop_LLM_prosody_PhoBert

# 2. Cài đặt dependencies
!pip install lightning==2.0.0 transformers>=4.30.0 phonemizer underthesea num2words librosa soundfile praat-parselmouth tensorboard einops
!pip install -r requirements.txt

# 3. Tạo output directory
import os
os.makedirs("/kaggle/working/outputs", exist_ok=True)
print("✅ Đã setup xong!")


In [ ]:
# ⚠️ CLEANUP: Remove any accidentally created nested IPIATTS directories
import os
import shutil

# Check for nested IPIATTS dirs caused by wrong path detection
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

# List contents to check for nested dirs
contents = os.listdir(cwd)
print(f"Contents: {contents}")

# Remove problematic nested paths if they exist
if "IPIATTS" in contents:
    ipiatts_path = os.path.join(cwd, "IPIATTS")
    if os.path.isdir(ipiatts_path):
        print(f"⚠️ Found nested IPIATTS directory at {ipiatts_path}")
        try:
            shutil.rmtree(ipiatts_path)
            print("✅ Removed nested IPIATTS directory")
        except Exception as e:
            print(f"❌ Could not remove: {e}")

# Also check disk usage
os.system("df -h /kaggle/working 2>/dev/null || echo 'Checking disk space...'")
print("✅ Cleanup complete")


## 2. Fix Lightning + Espeak

In [ ]:
# Fix lỗi Lightning
!pip uninstall -y lightning lightning-cloud
!pip install lightning==2.1.0 lightning-cloud==0.5.37

# Cài đặt espeak-ng
!apt-get update
!apt-get install -y espeak-ng
print("✅ Đã fix Lightning và espeak-ng!")


## 3. Cập nhật Symbols, Cleaners và DataModule

In [ ]:
# Cập nhật symbols.py với tiếng Việt đầy đủ
symbols_code = """
# -*- coding: utf-8 -*-

_pad = "_"
_punctuation = [
    chr(35),   # #
    chr(59),   # ;
    chr(58),   # :
    chr(44),   # ,
    chr(46),   # .
    chr(33),   # !
    chr(63),   # ?
    chr(161),  # ¡
    chr(191),  # ¿
    chr(45),   # -
    chr(8212), # —
    chr(8230), # …
    chr(39),   # '
    chr(34),   # "
    chr(171),  # «
    chr(187),  # »
    chr(8220), # "
    chr(8221), # "
    chr(40),   # (
    chr(41),   # )
    chr(91),   # [
    chr(93),   # ]
    chr(47),   # /
    chr(37),   # %
    chr(32),   # space
]

_letters_lower = list("abcdefghijklmnopqrstuvwxyzđ")
_vietnamese_accents_lower = [
    "à", "á", "ả", "ã", "ạ",
    "ă", "ằ", "ắ", "ẳ", "ẵ", "ặ",
    "â", "ầ", "ấ", "ẩ", "ẫ", "ậ",
    "è", "é", "ẻ", "ẽ", "ẹ",
    "ê", "ề", "ế", "ể", "ễ", "ệ",
    "ì", "í", "ỉ", "ĩ", "ị",
    "ò", "ó", "ỏ", "õ", "ọ",
    "ô", "ồ", "ố", "ổ", "ỗ", "ộ",
    "ơ", "ờ", "ớ", "ở", "ỡ", "ợ",
    "ù", "ú", "ủ", "ũ", "ụ",
    "ư", "ừ", "ứ", "ử", "ữ", "ự",
    "ỳ", "ý", "ỷ", "ỹ", "ỵ",
]

_letters_upper = list("ABCDEFGHIJKLMNOPQRSTUVWXYZĐ")
_vietnamese_accents_upper = [
    "À", "Á", "Ả", "Ã", "Ạ",
    "Ă", "Ằ", "Ắ", "Ẳ", "Ẵ", "Ặ",
    "Â", "Ầ", "Ấ", "Ẩ", "Ẫ", "Ậ",
    "È", "É", "Ẻ", "Ẽ", "Ẹ",
    "Ê", "Ề", "Ế", "Ể", "Ễ", "Ệ",
    "Ì", "Í", "Ỉ", "Ĩ", "Ị",
    "Ò", "Ó", "Ỏ", "Õ", "Ọ",
    "Ô", "Ồ", "Ố", "Ổ", "Ỗ", "Ộ",
    "Ơ", "Ờ", "Ớ", "Ở", "Ỡ", "Ợ",
    "Ù", "Ú", "Ủ", "Ũ", "Ụ",
    "Ư", "Ừ", "Ứ", "Ử", "Ữ", "Ự",
    "Ỳ", "Ý", "Ỷ", "Ỹ", "Ỵ",
]

_digits = list("0123456789")

_letters_ipa = [
    chr(593), chr(592), chr(594), chr(230), chr(595), chr(665), chr(946),
    chr(596), chr(597), chr(231), chr(599), chr(598), chr(240), chr(676),
    chr(601), chr(600), chr(602), chr(603), chr(604), chr(605), chr(606),
    chr(607), chr(644), chr(609), chr(608), chr(610), chr(667), chr(614),
    chr(615), chr(295), chr(613), chr(668), chr(616), chr(618), chr(669),
    chr(621), chr(620), chr(619), chr(622), chr(671), chr(625), chr(623),
    chr(624), chr(331), chr(627), chr(626), chr(628), chr(248), chr(629),
    chr(632), chr(952), chr(339), chr(630), chr(664), chr(633), chr(634),
    chr(638), chr(635), chr(640), chr(641), chr(637), chr(642), chr(643),
    chr(648), chr(649), chr(650), chr(651), chr(11377), chr(652), chr(611),
    chr(612), chr(653), chr(967), chr(654), chr(655), chr(657), chr(656),
    chr(658), chr(660), chr(673), chr(661), chr(674), chr(448), chr(449),
    chr(450), chr(451), chr(712), chr(716), chr(720), chr(721), chr(700),
    chr(692), chr(688), chr(689), chr(690), chr(695), chr(736), chr(740),
    chr(734), chr(8595), chr(8593), chr(8594), chr(8599), chr(8600),
    chr(809), chr(7547), chr(810), chr(865)
]

symbols = (
    [_pad]
    + _punctuation
    + _letters_lower
    + _letters_upper
    + _vietnamese_accents_lower
    + _vietnamese_accents_upper
    + _digits
    + _letters_ipa
)

SPACE_ID = symbols.index(" ")
_symbol_to_id = {s: i for i, s in enumerate(symbols)}
_id_to_symbol = {i: s for i, s in enumerate(symbols)}
"""

import os
import sys

# Workspace path - use current working directory
workspace_root = os.getcwd()
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

# Ghi symbols.py vào đúng vị trí
symbols_path = os.path.join(workspace_root, "matcha", "text", "symbols.py")
os.makedirs(os.path.dirname(symbols_path), exist_ok=True)

with open(symbols_path, "w", encoding="utf-8") as f:
    f.write(symbols_code)

print(f"✅ Đã cập nhật symbols.py tại: {symbols_path}")


## 4. Cập nhật Cleaners (Training vs Inference)

In [ ]:
# Cập nhật cleaners.py cho TRAINING (input đã là IPA) và INFERENCE (text -> IPA)
cleaner_content = """
import re
import sys

try:
    from underthesea import text_normalize
    from phonemizer.backend import EspeakBackend
    from num2words import num2words
except ImportError:
    pass

_ESPEAK = None
try:
    _ESPEAK = EspeakBackend(
        "vi",
        preserve_punctuation=True,
        language_switch="remove-flags", 
        with_stress=True,               
        tie=True
    )
except Exception:
    pass

_RE_NUMBER = re.compile(r"\d+")

def basic_cleaners_phothong(text):
    '''
    Cleaner dùng cho TRAINING (input đã là IPA).
    Chỉ chuẩn hóa khoảng trắng, KHÔNG gọi espeak.
    '''
    if text is None: 
        return ""
    return re.sub(r"\s+", " ", text).strip()

def vietnamese_text_to_ipa(text):
    '''
    Cleaner dùng cho INFERENCE (text thường -> IPA).
    '''
    if _ESPEAK is None:
        raise RuntimeError("Lỗi: Chưa cài espeak-ng hoặc phonemizer!")

    text = text_normalize(text)
    
    def _replace_number(match):
        try:
            return num2words(int(match.group()), lang='vi')
        except:
            return match.group()
            
    text = _RE_NUMBER.sub(_replace_number, text)
    text = text.replace("%", " phần trăm").replace("&", " và ").replace("+", " cộng ")
    ipa = _ESPEAK.phonemize(text, strip=True)
    ipa = re.sub(r'\(.*?\)', '', ipa)
    ipa = re.sub(r"\s+", " ", ipa).strip()
    
    return ipa
"""

import os

# Ghi cleaners.py vào đúng vị trí
workspace_root = os.getcwd()
cleaners_path = os.path.join(workspace_root, "matcha", "text", "cleaners.py")
os.makedirs(os.path.dirname(cleaners_path), exist_ok=True)

with open(cleaners_path, "w", encoding="utf-8") as f:
    f.write(cleaner_content)

print(f"✅ Đã cập nhật cleaners.py tại: {cleaners_path}")


## 5. Cập nhật TextMelDataModule

In [ ]:
# Cập nhật TextMelDataModule để đọc 3 cột và tính thêm pitch/energy cho prosody
# Raw_text sẽ được dùng cho PhoBERT (Prosody Analysis)

datamodule_code = """
import random
from pathlib import Path
from typing import Optional
import torch
import torchaudio as ta
from torch.nn import functional as F
from lightning import LightningDataModule
from torch.utils.data.dataloader import DataLoader
from matcha.text import text_to_sequence
from matcha.utils.audio import mel_spectrogram
from matcha.utils.model import fix_len_compatibility, normalize
from matcha.utils.utils import intersperse


def parse_filelist(filelist_path, split_char="|"):
    with open(filelist_path, encoding="utf-8") as f:
        filepaths_and_text = [line.strip().split(split_char) for line in f]
    return filepaths_and_text


def pad_or_trim(feature: torch.Tensor, target_len: int) -> torch.Tensor:
    # Pad hoặc cắt feature 1D về đúng số frame mel
    if feature.shape[-1] < target_len:
        feature = F.pad(feature, (0, target_len - feature.shape[-1]))
    elif feature.shape[-1] > target_len:
        feature = feature[..., :target_len]
    return feature


class TextMelDataModule(LightningDataModule):
    def __init__(
        self,
        name,
        train_filelist_path,
        valid_filelist_path,
        batch_size,
        num_workers,
        pin_memory,
        cleaners,
        add_blank,
        n_spks,
        n_fft,
        n_feats,
        sample_rate,
        hop_length,
        win_length,
        f_min,
        f_max,
        data_statistics,
        seed,
        load_durations,
        audio_root=None,
    ):
        super().__init__()
        self.save_hyperparameters(logger=False)

    def setup(self, stage: Optional[str] = None):
        self.trainset = TextMelDataset(
            self.hparams.train_filelist_path,
            self.hparams.n_spks,
            self.hparams.cleaners,
            self.hparams.add_blank,
            self.hparams.n_fft,
            self.hparams.n_feats,
            self.hparams.sample_rate,
            self.hparams.hop_length,
            self.hparams.win_length,
            self.hparams.f_min,
            self.hparams.f_max,
            self.hparams.data_statistics,
            self.hparams.seed,
            self.hparams.load_durations,
            audio_root=self.hparams.audio_root,
        )
        self.validset = TextMelDataset(
            self.hparams.valid_filelist_path,
            self.hparams.n_spks,
            self.hparams.cleaners,
            self.hparams.add_blank,
            self.hparams.n_fft,
            self.hparams.n_feats,
            self.hparams.sample_rate,
            self.hparams.hop_length,
            self.hparams.win_length,
            self.hparams.f_min,
            self.hparams.f_max,
            self.hparams.data_statistics,
            self.hparams.seed,
            self.hparams.load_durations,
            audio_root=self.hparams.audio_root,
        )

    def train_dataloader(self):
        return DataLoader(
            self.trainset,
            batch_size=self.hparams.batch_size,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            shuffle=True,
            collate_fn=TextMelBatchCollate(self.hparams.n_spks),
        )

    def val_dataloader(self):
        return DataLoader(
            self.validset,
            batch_size=self.hparams.batch_size,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            shuffle=False,
            collate_fn=TextMelBatchCollate(self.hparams.n_spks),
        )


class TextMelDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        filelist_path,
        n_spks,
        cleaners,
        add_blank=True,
        n_fft=1024,
        n_mels=80,
        sample_rate=22050,
        hop_length=256,
        win_length=1024,
        f_min=0.0,
        f_max=8000,
        data_parameters=None,
        seed=None,
        load_durations=False,
        audio_root=None,
    ):
        self.filepaths_and_text = parse_filelist(filelist_path)
        self.n_spks = n_spks
        self.cleaners = cleaners
        self.add_blank = add_blank
        self.n_fft = n_fft
        self.n_mels = n_mels
        self.sample_rate = sample_rate
        self.hop_length = hop_length
        self.win_length = win_length
        self.f_min = f_min
        self.f_max = f_max
        self.load_durations = load_durations
        self.audio_root = Path(audio_root) if audio_root else None
        self.data_parameters = data_parameters if data_parameters else {
            "mel_mean": 0,
            "mel_std": 1,
            "pitch_mean": 0,
            "pitch_std": 1,
            "energy_mean": 0,
            "energy_std": 1,
        }
        random.seed(seed)
        random.shuffle(self.filepaths_and_text)

    def load_audio(self, filepath: str) -> torch.Tensor:
        filepath = Path(filepath)
        if self.audio_root and not filepath.is_absolute():
            filepath = self.audio_root / filepath
        audio, sr = ta.load(filepath)
        if sr != self.sample_rate:
            audio = ta.functional.resample(audio, sr, self.sample_rate)
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)
        return audio

    def get_datapoint(self, filepath_and_text):
        if len(filepath_and_text) >= 3:
            filepath = filepath_and_text[0]
            raw_text = filepath_and_text[1]
            ipa_text = filepath_and_text[2]
        else:
            filepath = filepath_and_text[0]
            raw_text = filepath_and_text[1]
            ipa_text = filepath_and_text[1]

        text, cleaned_text = self.get_text(ipa_text, add_blank=self.add_blank)
        audio = self.load_audio(filepath)
        mel = self.get_mel(audio)
        pitch, energy = self.get_pitch_energy(audio, mel.shape[-1])
        durations = self.get_durations(filepath, text) if self.load_durations else None

        return {
            "x": text,
            "y": mel,
            "pitch": pitch,
            "energy": energy,
            "spk": 0,
            "filepath": filepath,
            "raw_text": raw_text,
            "durations": durations,
        }

    def get_mel(self, audio: torch.Tensor):
        mel = mel_spectrogram(
            audio,
            self.n_fft,
            self.n_mels,
            self.sample_rate,
            self.hop_length,
            self.win_length,
            self.f_min,
            self.f_max,
            center=False,
        ).squeeze()
        mel = normalize(mel, self.data_parameters.get("mel_mean", 0), self.data_parameters.get("mel_std", 1))
        return mel

    def get_pitch_energy(self, audio: torch.Tensor, target_len: int):
        mono = audio
        # Fix: detect_pitch_frequency không có hop_length parameter, dùng frame_time
        frame_time = self.hop_length / self.sample_rate
        pitch = ta.functional.detect_pitch_frequency(
            mono,
            sample_rate=self.sample_rate,
            frame_time=frame_time,
        ).squeeze(0)
        pitch = torch.log1p(pitch)
        pitch = torch.nan_to_num(pitch, nan=0.0, posinf=0.0, neginf=0.0)
        pitch = pad_or_trim(pitch, target_len)
        pitch = normalize(pitch, self.data_parameters.get("pitch_mean", 0), self.data_parameters.get("pitch_std", 1))

        frames = mono.unfold(1, self.win_length, self.hop_length)
        energy = torch.sqrt(torch.mean(frames ** 2, dim=-1)).squeeze(0)
        energy = torch.nan_to_num(energy, nan=0.0, posinf=0.0, neginf=0.0)
        energy = pad_or_trim(energy, target_len)
        energy = normalize(energy, self.data_parameters.get("energy_mean", 0), self.data_parameters.get("energy_std", 1))
        return pitch, energy

    def get_text(self, text, add_blank=True):
        text_norm, cleaned_text = text_to_sequence(text, self.cleaners)
        if self.add_blank:
            text_norm = intersperse(text_norm, 0)
        text_norm = torch.IntTensor(text_norm)
        return text_norm, cleaned_text

    def get_durations(self, filepath, text):
        return None

    def __getitem__(self, index):
        return self.get_datapoint(self.filepaths_and_text[index])

    def __len__(self):
        return len(self.filepaths_and_text)


class TextMelBatchCollate:
    def __init__(self, n_spks):
        self.n_spks = n_spks

    def __call__(self, batch):
        B = len(batch)
        y_max_length = max([item["y"].shape[-1] for item in batch])
        y_max_length = fix_len_compatibility(y_max_length)
        x_max_length = max([item["x"].shape[-1] for item in batch])
        n_feats = batch[0]["y"].shape[-2]

        y = torch.zeros((B, n_feats, y_max_length), dtype=torch.float32)
        x = torch.zeros((B, x_max_length), dtype=torch.long)
        pitch = torch.zeros((B, y_max_length), dtype=torch.float32)
        energy = torch.zeros((B, y_max_length), dtype=torch.float32)
        y_lengths, x_lengths = [], []
        raw_texts = []
        filepaths = []

        for i, item in enumerate(batch):
            y_, x_ = item["y"], item["x"]
            pitch_, energy_ = item["pitch"], item["energy"]
            y_lengths.append(y_.shape[-1])
            x_lengths.append(x_.shape[-1])
            y[i, :, : y_.shape[-1]] = y_
            x[i, : x_.shape[-1]] = x_
            pitch[i, : pitch_.shape[-1]] = pitch_
            energy[i, : energy_.shape[-1]] = energy_
            raw_texts.append(item["raw_text"])
            filepaths.append(item["filepath"])

        y_lengths = torch.tensor(y_lengths, dtype=torch.long)
        x_lengths = torch.tensor(x_lengths, dtype=torch.long)

        return {
            "x": x,
            "x_lengths": x_lengths,
            "y": y,
            "y_lengths": y_lengths,
            "pitch": pitch,
            "energy": energy,
            "spks": None,
            "raw_texts": raw_texts,
            "durations": None,
        }
"""

import os

# Ghi TextMelDataModule vào đúng vị trí
workspace_root = os.getcwd()
datamodule_path = os.path.join(workspace_root, "matcha", "data", "text_mel_datamodule.py")
os.makedirs(os.path.dirname(datamodule_path), exist_ok=True)

with open(datamodule_path, "w", encoding="utf-8") as f:
    f.write(datamodule_code)

print(f"✅ Đã cập nhật TextMelDataModule tại: {datamodule_path}")

## 6. Xử lý và chuẩn bị Filelists

In [ ]:
# Xác định đường dẫn input files
AUDIO_DIR = "/kaggle/input/data-audio-ipa/kaggle/working/data/subs_add_con"
TRAIN_LIST_INPUT = "/kaggle/input/text-ipa/audio_text_train_filelist_new_ipa.txt"
VAL_LIST_INPUT = "/kaggle/input/text-ipa/audio_text_val_filelist_new_ipa.txt"

# Đường dẫn output (nơi lưu filelist đã xử lý)
TRAIN_LIST_OUTPUT = "/kaggle/working/fixed_train.txt"
VAL_LIST_OUTPUT = "/kaggle/working/fixed_val.txt"

print(f"Audio source: {AUDIO_DIR}")
print(f"Train list: {TRAIN_LIST_INPUT} -> {TRAIN_LIST_OUTPUT}")
print(f"Val list: {VAL_LIST_INPUT} -> {VAL_LIST_OUTPUT}")


## 7. Fix File Paths

In [ ]:
import os

def fix_file_paths(input_file, output_file, audio_folder_path):
    """Fix paths trong filelist để phù hợp với Kaggle environment"""
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 3:
            # Tách filename từ path
            filename = parts[0].replace('\\', '/').split('/')[-1].replace('"', '')
            # Lấy text thường (cột 2)
            text = parts[1].replace('"', '').strip()
            # Lấy IPA (cột 3)
            ipa = parts[2].replace('"', '').strip()
            
            # Lưu: filename|text|ipa (TextMelDataModule sẽ tự nối với audio_root)
            new_lines.append(f"{filename}|{text}|{ipa}")
            
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("\n".join(new_lines))
    
    print(f"✅ Processed {len(new_lines)} lines -> {output_file}")

# Fix cả train và val filelists
fix_file_paths(TRAIN_LIST_INPUT, TRAIN_LIST_OUTPUT, AUDIO_DIR)
fix_file_paths(VAL_LIST_INPUT, VAL_LIST_OUTPUT, AUDIO_DIR)


## 8. Tính Mel Statistics

In [ ]:
import torch
import torchaudio
from tqdm.auto import tqdm

# Cấu hình mel spectrogram
sample_rate = 22050
n_fft = 1024
n_mels = 80
hop_length = 256
win_length = 1024
f_min = 0
f_max = 8000

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=sample_rate, n_fft=n_fft, n_mels=n_mels, 
    hop_length=hop_length, win_length=win_length, 
    f_min=f_min, f_max=f_max, center=False, power=1.0, 
    norm="slaney", mel_scale="slaney"
)

mel_sum = 0
mel_sq_sum = 0
total_frames = 0

print("⏳ Đang tính toán Mel Statistics...")
with open(TRAIN_LIST_OUTPUT, 'r', encoding='utf-8') as f:
    lines = f.readlines()[:500]  # Lấy 500 files đầu để tính nhanh

for line in tqdm(lines):
    filename = line.split('|')[0]
    audio_path = os.path.join(AUDIO_DIR, filename)
    try:
        wav, sr = torchaudio.load(audio_path)
        if sr != sample_rate:
            wav = torchaudio.functional.resample(wav, sr, sample_rate)
        
        mel = mel_transform(wav)
        mel = torch.log(torch.clamp(mel, min=1e-5))
        
        mel_sum += torch.sum(mel)
        mel_sq_sum += torch.sum(mel ** 2)
        total_frames += mel.numel()
    except Exception as e:
        continue

# Tính Mean và Std
dataset_mean = (mel_sum / total_frames).item()
dataset_std = torch.sqrt((mel_sq_sum / total_frames) - (dataset_mean ** 2)).item()

print(f"\n✅ KẾT QUẢ MEL STATISTICS:")
print(f"MEAN: {dataset_mean:.4f}")
print(f"STD:  {dataset_std:.4f}")

# Lưu để dùng sau
CALCULATED_MEAN = dataset_mean
CALCULATED_STD = dataset_std


In [ ]:
# Tính nhanh pitch/energy statistics (subset 300 files để tiết kiệm thời gian)
import os
import torch
import torchaudio
from tqdm.auto import tqdm

# Đảm bảo các tham số đã được define (lấy từ cell trước)
if 'sample_rate' not in locals():
    sample_rate = 22050
if 'win_length' not in locals():
    win_length = 1024
if 'hop_length' not in locals():
    hop_length = 256

pitch_sum = pitch_sq_sum = 0.0
energy_sum = energy_sq_sum = 0.0
pitch_frames = energy_frames = 0

print("⏳ Đang tính Pitch/Energy Statistics trên 300 files đầu...")

with open(TRAIN_LIST_OUTPUT, 'r', encoding='utf-8') as f:
    lines = f.readlines()[:300]  # Giới hạn để nhanh hơn trên Kaggle

success_count = 0
error_count = 0

for idx, line in enumerate(tqdm(lines)):
    filename = line.split('|')[0].strip()
    audio_path = os.path.join(AUDIO_DIR, filename)
    
    try:
        if not os.path.exists(audio_path):
            error_count += 1
            continue
            
        wav, sr = torchaudio.load(audio_path)
        if sr != sample_rate:
            wav = torchaudio.functional.resample(wav, sr, sample_rate)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        # Pitch - sử dụng detect_pitch_frequency với API đúng (không có hop_length)
        # frame_time = hop_length / sample_rate để tương đương
        frame_time = hop_length / sample_rate  # ~0.0116s with hop=256, sr=22050
        pitch = torchaudio.functional.detect_pitch_frequency(
            wav, 
            sample_rate=sample_rate,
            frame_time=frame_time,
        ).squeeze(0)
        
        pitch = torch.log1p(pitch)
        pitch = torch.nan_to_num(pitch, nan=0.0, posinf=0.0, neginf=0.0)
        # Only count non-zero values
        pitch_nonzero = pitch[pitch > 0]
        if len(pitch_nonzero) > 0:
            pitch_sum += torch.sum(pitch_nonzero)
            pitch_sq_sum += torch.sum(pitch_nonzero ** 2)
            pitch_frames += len(pitch_nonzero)

        # Energy - tính RMS energy per frame
        frames = wav.unfold(1, win_length, hop_length)
        energy = torch.sqrt(torch.mean(frames ** 2, dim=-1)).squeeze(0)
        energy = torch.nan_to_num(energy, nan=0.0, posinf=0.0, neginf=0.0)
        # Only count non-zero values
        energy_nonzero = energy[energy > 0]
        if len(energy_nonzero) > 0:
            energy_sum += torch.sum(energy_nonzero)
            energy_sq_sum += torch.sum(energy_nonzero ** 2)
            energy_frames += len(energy_nonzero)
        
        success_count += 1
    except Exception as e:
        error_count += 1
        if idx < 3:
            print(f"[{idx}] Error: {e}")
        continue

print(f"\n✅ Đã xử lý thành công {success_count}/{len(lines)} files")
print(f"❌ Lỗi: {error_count} files")

if pitch_frames > 0:
    pitch_mean = (pitch_sum / pitch_frames).item()
    pitch_std = torch.sqrt((pitch_sq_sum / pitch_frames) - (pitch_mean ** 2)).item()
    print(f"✅ Pitch mean/std: {pitch_mean:.4f} / {pitch_std:.4f}")
else:
    print("⚠️ Không tính được pitch, dùng mặc định")
    pitch_mean, pitch_std = 0.0, 1.0

if energy_frames > 0:
    energy_mean = (energy_sum / energy_frames).item()
    energy_std = torch.sqrt((energy_sq_sum / energy_frames) - (energy_mean ** 2)).item()
    print(f"✅ Energy mean/std: {energy_mean:.4f} / {energy_std:.4f}")
else:
    print("⚠️ Không tính được energy, dùng mặc định")
    energy_mean, energy_std = 0.0, 1.0

PITCH_MEAN, PITCH_STD = pitch_mean, pitch_std
ENERGY_MEAN, ENERGY_STD = energy_mean, energy_std

## 9. Tạo Training Script

In [ ]:
# ============================================================================
# ✅ KIỂM TRA BIẾN TRƯỚC KHI TẠO SCRIPT
# ============================================================================
import os

print("="*80)
print("🔍 KIỂM TRA CÁC BIẾN CẦN THIẾT CHO TRAINING SCRIPT")
print("="*80)

# Kiểm tra và định nghĩa các biến nếu chưa tồn tại
try:
    print(f"✅ TRAIN_LIST_OUTPUT = {TRAIN_LIST_OUTPUT}")
except NameError:
    print("⚠️  TRAIN_LIST_OUTPUT chưa được định nghĩa, dùng mặc định")
    TRAIN_LIST_OUTPUT = "/kaggle/working/fixed_train.txt"
    print(f"   → {TRAIN_LIST_OUTPUT}")

try:
    print(f"✅ VAL_LIST_OUTPUT = {VAL_LIST_OUTPUT}")
except NameError:
    print("⚠️  VAL_LIST_OUTPUT chưa được định nghĩa, dùng mặc định")
    VAL_LIST_OUTPUT = "/kaggle/working/fixed_val.txt"
    print(f"   → {VAL_LIST_OUTPUT}")

try:
    print(f"✅ AUDIO_DIR = {AUDIO_DIR}")
except NameError:
    print("⚠️  AUDIO_DIR chưa được định nghĩa, dùng mặc định")
    AUDIO_DIR = "/kaggle/input/data-audio-ipa/kaggle/working/data/subs_add_con"
    print(f"   → {AUDIO_DIR}")

try:
    print(f"✅ CALCULATED_MEAN = {CALCULATED_MEAN:.4f}")
except NameError:
    print("⚠️  CALCULATED_MEAN chưa được tính, dùng mặc định")
    CALCULATED_MEAN = -4.5129
    print(f"   → {CALCULATED_MEAN:.4f}")

try:
    print(f"✅ CALCULATED_STD = {CALCULATED_STD:.4f}")
except NameError:
    print("⚠️  CALCULATED_STD chưa được tính, dùng mặc định")
    CALCULATED_STD = 2.3453
    print(f"   → {CALCULATED_STD:.4f}")

print("="*80)
print("✅ Tất cả biến đã được định nghĩa, sẵn sàng tạo training script!")
print("="*80)

In [ ]:
# ============================================================================
# TẠO TRAINING SCRIPT
# ============================================================================

script = f"""
import sys
import os
import torch
from argparse import Namespace

# ============================================================================
# TRAINING SCRIPT - MATCHA-TTS WITH LLM PROSODY
# ============================================================================

from lightning import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.strategies import DDPStrategy
from matcha.models.matcha_tts import MatchaTTS
from matcha.data.text_mel_datamodule import TextMelDataModule
from matcha.text.symbols import symbols
import torch.serialization

# Fix pickle
torch.serialization.add_safe_globals([Namespace])

# ============================================================================
# CONFIG TRAINING
# ============================================================================
CONFIG = {{
    "train_filelist": "{TRAIN_LIST_OUTPUT}",
    "val_filelist": "{VAL_LIST_OUTPUT}",
    "output_dir": "outputs/matcha_prosody_fixed",
    
    # Model
    "n_spks": 1,
    "spk_emb_dim": 64,
    "n_feats": 80,
    
    # Prosody
    "llm_model_name": "vinai/phobert-base",
    "prosody_dim": 192,  # Giảm từ 256 → 192
    "use_token_level_prosody": True,
    "finetune_llm": False,  # ✅ TẮT fine-tune LLM để tiết kiệm memory
    
    # Training - OPTIMIZED FOR MEMORY
    "batch_size": 4,  # ✅ Giảm từ 12 → 4
    "learning_rate": 5e-5,
    "max_epochs": 200,
    "num_workers": 2,  # ✅ Giảm từ 4 → 2
    "accumulate_grad_batches": 4,  # ✅ Tăng từ 2 → 4 (effective batch = 4*4*2 = 32)
    
    # Hardware
    "accelerator": "gpu",
    "devices": 2,
    "precision": "16-mixed",
    
    # Paths
    "audio_root": "{AUDIO_DIR}",
    "resume_from_checkpoint": None,
}}

CONFIG["n_vocab"] = len(symbols) + 1

# Data stats
DATA_STATS = {{
    "mel_mean": {CALCULATED_MEAN},
    "mel_std": {CALCULATED_STD},
}}

# ============================================================================
# ENCODER CONFIG - OPTIMIZED FOR MEMORY
# ============================================================================
ENCODER_CONFIG = {{
    "encoder_type": "RoPE Encoder",
    "encoder_params": {{
        "n_feats": CONFIG["n_feats"],
        "n_channels": 512,  # ✅ Giảm từ 768 → 512
        "filter_channels": 1536,  # ✅ Giảm từ 2048 → 1536
        "filter_channels_dp": 384,  # ✅ Giảm từ 512 → 384
        "n_heads": 8,  # ✅ Giảm từ 12 → 8
        "n_layers": 8,  # ✅ Giảm từ 12 → 8
        "kernel_size": 3,
        "p_dropout": 0.1,
        "spk_emb_dim": CONFIG["spk_emb_dim"],
        "n_spks": CONFIG["n_spks"],
        "prenet": True,
    }},
    "duration_predictor_params": {{
        "filter_channels_dp": 256,  # ✅ Giảm từ 384 → 256
        "kernel_size": 3,
        "p_dropout": 0.1,
    }},
}}

# ============================================================================
# DECODER CONFIG - OPTIMIZED FOR MEMORY
# ============================================================================
DECODER_CONFIG = {{
    "channels": [256, 256],  # ✅ Giảm từ [384,384] → [256,256]
    "dropout": 0.2,
    "attention_head_dim": 64,
    "n_blocks": 4,  # ✅ Giảm từ 6 → 4
    "num_mid_blocks": 4,  # ✅ Giảm từ 6 → 4
    "num_heads": 8,  # ✅ Giảm từ 12 → 8
    "act_fn": "gelu",
}}

CFM_CONFIG = {{
    "sigma_min": 1e-4,
    "solver": "euler",
    "t_scheduler": "cosine",
}}

print("="*80)
print("🍵 MATCHA-TTS TRAINING")
print("="*80)
print("✅ Batch Size: {{}} x {{}} = {{}} (effective)".format(
    CONFIG['batch_size'], 
    CONFIG['accumulate_grad_batches'], 
    CONFIG['batch_size'] * CONFIG['accumulate_grad_batches']
))
print("✅ Learning Rate: {{}}".format(CONFIG['learning_rate']))
print("✅ Fine-tune LLM: {{}}".format(CONFIG['finetune_llm']))
print("="*80)

def create_model():
    # Create encoder
    encoder = Namespace(
        encoder_type=ENCODER_CONFIG["encoder_type"],
        encoder_params=Namespace(**ENCODER_CONFIG["encoder_params"]),
        duration_predictor_params=Namespace(**ENCODER_CONFIG["duration_predictor_params"]),
    )
    
    # Decoder and CFM
    decoder = DECODER_CONFIG.copy()
    cfm = Namespace(**CFM_CONFIG)
    
    # Create model
    model = MatchaTTS(
        n_vocab=CONFIG["n_vocab"],
        n_spks=CONFIG["n_spks"],
        spk_emb_dim=CONFIG["spk_emb_dim"],
        n_feats=CONFIG["n_feats"],
        encoder=encoder,
        decoder=decoder,
        cfm=cfm,
        data_statistics=DATA_STATS,
        out_size=None,
        optimizer=torch.optim.AdamW,
        optimizer_kwargs={{"lr": CONFIG["learning_rate"], "weight_decay": 1e-6}},
        llm_model_name=CONFIG["llm_model_name"],
        prosody_dim=CONFIG["prosody_dim"],
        use_token_level_prosody=CONFIG["use_token_level_prosody"],
        finetune_llm=CONFIG["finetune_llm"],
    )
    
    return model

def create_datamodule():
    datamodule = TextMelDataModule(
        name="matcha_vi",
        train_filelist_path=CONFIG["train_filelist"],
        valid_filelist_path=CONFIG["val_filelist"],
        batch_size=CONFIG["batch_size"],
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
        cleaners=["basic_cleaners_phothong"],
        add_blank=True,
        n_spks=CONFIG["n_spks"],
        n_fft=1024,
        n_feats=CONFIG["n_feats"],
        sample_rate=22050,
        hop_length=256,
        win_length=1024,
        f_min=0,
        f_max=8000,
        data_statistics=DATA_STATS,
        seed=1234,
        load_durations=False,
        audio_root=CONFIG["audio_root"],
    )
    return datamodule

def train():
    # Init
    model = create_model()
    datamodule = create_datamodule()

    # Callbacks
    checkpoint_cb = ModelCheckpoint(
        dirpath=CONFIG["output_dir"] + "/checkpoints",
        filename="matcha-{{epoch:03d}}-{{loss/val_epoch:.4f}}",
        monitor="loss/val_epoch",
        mode="min",
        save_top_k=3,
        save_last=True,
    )
    
    lr_monitor = LearningRateMonitor(logging_interval="step")

    # Trainer with DDP
    strategy = DDPStrategy(find_unused_parameters=True)
    
    trainer = Trainer(
        accelerator=CONFIG["accelerator"],
        devices=CONFIG["devices"],
        strategy=strategy,
        max_epochs=CONFIG["max_epochs"],
        callbacks=[checkpoint_cb, lr_monitor],
        logger=TensorBoardLogger(CONFIG["output_dir"], name="logs"),
        precision=CONFIG["precision"],
        gradient_clip_val=1.0,
        accumulate_grad_batches=CONFIG["accumulate_grad_batches"],
        log_every_n_steps=50,
        val_check_interval=0.5,
    )

    # Train
    print("\\n🚀 Starting training...")
    trainer.fit(model, datamodule=datamodule)
    
    print("\\n" + "="*80)
    print("✅ TRAINING COMPLETED!")
    print("="*80)

if __name__ == "__main__":
    train()
"""

# Save script with absolute path
workspace_root = os.getcwd()
script_path = os.path.join(workspace_root, "train_matcha_final.py")

with open(script_path, "w") as f:
    f.write(script)

print(f"\n✅ Training script saved to: {script_path}")
print(f"📝 Run it with: !python {script_path}")

In [ ]:
# ============================================================================
# ✅ VALIDATION: Kiểm tra TextMelDataModule parameters
# ============================================================================
import inspect
from matcha.data.text_mel_datamodule import TextMelDataModule

print("="*80)
print("🔍 KIỂM TRA THAM SỐ TextMelDataModule")
print("="*80)

# Lấy signature của __init__
sig = inspect.signature(TextMelDataModule.__init__)
required_params = []
optional_params = []

for param_name, param in sig.parameters.items():
    if param_name == 'self':
        continue
    if param.default == inspect.Parameter.empty:
        required_params.append(param_name)
    else:
        optional_params.append(f"{param_name}={param.default}")

print("\n📋 Required parameters:")
for p in required_params:
    print(f"  ✓ {p}")

print("\n📋 Optional parameters:")
for p in optional_params[:5]:  # Show first 5
    print(f"  • {p}")
if len(optional_params) > 5:
    print(f"  ... and {len(optional_params) - 5} more")

print("\n" + "="*80)
print("✅ Validation complete - sẽ dùng đúng tham số này trong training script")
print("="*80)


In [ ]:
# ============================================================================
# ✅ KIỂM TRA: Stats đã được tính chưa?
# ============================================================================

print("="*80)
print("🔍 KIỂM TRA CÁC STATS ĐÃ TÍNH")
print("="*80)

# Kiểm tra Mel Stats
try:
    print(f"✅ CALCULATED_MEAN = {CALCULATED_MEAN:.4f}")
    print(f"✅ CALCULATED_STD = {CALCULATED_STD:.4f}")
    mel_stats_ok = True
except NameError as e:
    print(f"❌ LỖI MEL STATS: {e}")
    mel_stats_ok = False

# Kiểm tra Pitch/Energy Stats
try:
    print(f"✅ PITCH_MEAN = {PITCH_MEAN:.4f}")
    print(f"✅ PITCH_STD = {PITCH_STD:.4f}")
    print(f"✅ ENERGY_MEAN = {ENERGY_MEAN:.4f}")
    print(f"✅ ENERGY_STD = {ENERGY_STD:.4f}")
    prosody_stats_ok = True
except NameError as e:
    print(f"❌ LỖI PROSODY STATS: {e}")
    prosody_stats_ok = False

print("="*80)

if not (mel_stats_ok and prosody_stats_ok):
    print("⚠️  CẢNH BÁO: Một số stats chưa được tính!")
    print("✅ Dùng giá trị mặc định (tạm thời)")
    
    if not mel_stats_ok:
        CALCULATED_MEAN = -4.5129
        CALCULATED_STD = 2.3453
        
    if not prosody_stats_ok:
        PITCH_MEAN = 5.0552
        PITCH_STD = 0.6101
        ENERGY_MEAN = 0.1009
        ENERGY_STD = 0.0771

print(f"\n✅ Final Stats:")
print(f"   Mel: Mean={CALCULATED_MEAN:.4f}, Std={CALCULATED_STD:.4f}")
print(f"   Pitch: Mean={PITCH_MEAN:.4f}, Std={PITCH_STD:.4f}")
print(f"   Energy: Mean={ENERGY_MEAN:.4f}, Std={ENERGY_STD:.4f}")

## 🎯 Prosody Technique Được Áp Dụng

### 📋 Architecture:
```
Raw Vietnamese Text (Column 2)
        ↓
   [PhoBERT]  ← Lấy embeddings từ LLM
        ↓
Prosody Features (768-dim)
        ↓
[Projection Layer] → prosody_dim (256)
        ↓
[Prosody Fusion Module]
        ↓
Kết hợp với Text Encoder Output
        ↓
[CFM Decoder] ← Mel-spectrogram cuối cùng
```

### ✨ Key Components:

1. **LLMProsodyAnalyzer** (`matcha/models/components/prosody_analyzer.py`)
   - Dùng PhoBERT (vinai/phobert-base) để phân tích prosody
   - Extract toàn bộ 768-dim embeddings từ input text
   - Frozen LLM weights (không train)
   - Project xuống prosody_dim (256)

2. **ProsodyFusion** (`matcha/models/components/prosody_fusion.py`)
   - Cross-attention fusion giữa text encoder output và prosody features
   - Gating mechanism để kiểm soát ảnh hưởng của prosody
   - Output dimension = text encoder dimension

3. **Data Pipeline**
   - Input filelist: `filename|raw_text|ipa_text`
   - raw_text → PhoBERT → prosody embeddings
   - ipa_text → mel-spectrogram
   - Kết hợp cả hai qua fusion module

4. **Training Strategy**
   - Batch có chứa `raw_texts` để extract prosody on-the-fly
   - PhoBERT frozen, chỉ train fusion module + decoder
   - Joint learning của text synthesis + prosody conditioning

### 📊 Configuration:
- llm_model_name: `vinai/phobert-base`
- prosody_dim: 256 (đủ lớn để capture prosody features)
- fusion_type: attention-based với gating
- Prosody influence: controlled bằng learned gate

In [ ]:
# ============================================================================
# 📝 PROSODY USAGE EXAMPLES
# ============================================================================

print("""
🎯 PROSODY TECHNIQUE DETAILS:
================================

1. **Filelist Format (3 columns):**
   filename|raw_vietnamese_text|ipa_phonemes
   
   Example:
   audio_001.wav|xin chào các bạn|ks i n tʃ ə w ə ʔ i ə...
   
   Column 1: Filename (relative path)
   Column 2: RAW VIETNAMESE TEXT → input cho PhoBERT
   Column 3: IPA PHONEMES → input cho Matcha synthesis

2. **PhoBERT Prosody Analysis:**
   - Tokenize raw_text với PhoBERT tokenizer
   - Pass qua PhoBERT model → 768-dim embeddings
   - Project xuống 256-dim prosody_dim
   - Fusion với text encoder output qua attention

3. **Training Data Requirements:**
   ✓ Raw Vietnamese text MUST be provided (column 2)
   ✓ Text should be grammatically correct
   ✓ PhoBERT sẽ extract ngữ cảnh + prosody patterns
   
4. **Inference with Prosody:**
   model.synthesise(
       x=ipa_tokens,              # IPA tokenized
       x_lengths=seq_lengths,     # Token sequence lengths
       raw_texts=["xin chào"],    # RAW TEXT for prosody!
       n_timesteps=10             # Diffusion steps
   )

5. **Performance Impact:**
   - Adding prosody increases VRAM ~5% (768 embeddings/batch)
   - Inference ~10% slower (PhoBERT forward pass)
   - Quality improvement: ✓✓✓ (better intonation, stress, rhythm)

💡 TIP: Để tốt nhất, chuẩn bị raw_text sao cho:
   - Chính tả đúng (spelling matters for PhoBERT)
   - Dấu câu chính xác (periods, commas affect prosody)
   - Không viết tắt
""")


## 10. BẮT ĐẦU TRAINING

In [ ]:
# Cài đặt rootutils (required by matcha/train.py)
!pip install rootutils -q
print("✅ Đã cài rootutils!")


In [ ]:
# ============================================================================
# 🧹 CLEAN GPU MEMORY TRƯỚC KHI TRAIN
# ============================================================================
import gc
import torch

print("🧹 Cleaning GPU memory...")

# Clear Python garbage
gc.collect()

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    
    # Show memory stats
    for i in range(torch.cuda.device_count()):
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        free = total - allocated
        
        print(f"\n📊 GPU {i}:")
        print(f"  Total: {total:.2f} GB")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved: {reserved:.2f} GB")
        print(f"  Free: {free:.2f} GB")

print("\n✅ GPU memory cleaned!")
print("="*80)


In [ ]:
!python train_matcha_final.py

## 11. Kiểm tra text processing

In [ ]:
from matcha.text import cleaners
from matcha.text import symbols

print("🔤 Tổng số ký hiệu trong tokenizer:", len(symbols))
print(symbols[:50])

In [ ]:
# Utility: Inspect text length vs model limits and suggest chunking
import re
from typing import List, Tuple

from matcha.text import text_to_sequence
from matcha.text.cleaners import basic_cleaners_phothong

# Optional: check PhoBERT tokenizer limits if available
try:
    from transformers import AutoTokenizer, AutoModel
    _tok = AutoTokenizer.from_pretrained("vinai/phobert-base")
    _llm = AutoModel.from_pretrained("vinai/phobert-base")
    _special_tokens = _tok.num_special_tokens_to_add(pair=False)
    _llm_max_pos = getattr(_llm.config, "max_position_embeddings", 512)
    _llm_pos_actual = getattr(getattr(_llm, "embeddings", None), "position_embeddings", None)
    if _llm_pos_actual is not None:
        _llm_max_safe = max(1, _llm_pos_actual.num_embeddings - 1)
    else:
        _llm_max_safe = _llm_max_pos
except Exception:
    _tok = None
    _llm = None
    _special_tokens = 2
    _llm_max_safe = 512


def _sentence_split(text: str) -> List[str]:
    parts = re.split(r"([.!?])", text)
    # Re-attach punctuation
    sents = ["".join(parts[i:i+2]).strip() for i in range(0, len(parts), 2)]
    return [s for s in sents if s]


def inspect_text(text: str, phoneme_warn: int = 320) -> Tuple[int, int]:
    # Clean & phonemize to estimate phoneme sequence length
    cleaned = basic_cleaners_phothong(text)
    seq, _ = text_to_sequence(text, ["basic_cleaners_phothong"])  # uses cleaners internally
    phoneme_len = len(seq)

    # Tokenizer length (raw text) if available
    if _tok is not None:
        enc = _tok(text, truncation=False, add_special_tokens=True, return_tensors=None)
        llm_len = len(enc["input_ids"])  # includes specials
    else:
        llm_len = 0

    print(f"🔎 Phoneme length: {phoneme_len}")
    if _tok is not None:
        print(f"🔎 PhoBERT tokens (with specials): {llm_len} | safe max: {_llm_max_safe}")
    else:
        print("ℹ️ PhoBERT tokenizer unavailable; skipping LLM length check.")

    if phoneme_len > phoneme_warn:
        print("⚠️ Phoneme sequence is long; risk of high VRAM/OOM. Consider chunking.")
    if _tok is not None and llm_len > _llm_max_safe:
        print("⚠️ Raw text exceeds PhoBERT position limit; it will be truncated.")

    return phoneme_len, llm_len


def suggest_chunking(text: str) -> List[str]:
    sents = _sentence_split(text)
    if len(sents) <= 1:
        # Fallback: split by commas if no clear sentences
        sents = [t.strip() for t in re.split(r",", text) if t.strip()]
    print(f"🧩 Suggested {len(sents)} chunks for synthesis.")
    return sents

# Demo: replace with your paragraph
sample_text = "Xin chào các bạn, đây là một đoạn văn dài thử nghiệm để đánh giá khả năng xử lý văn bản của hệ thống. Hệ thống sẽ kiểm tra độ dài phoneme và giới hạn của PhoBERT."
inspect_text(sample_text)
chunks = suggest_chunking(sample_text)
for i, c in enumerate(chunks[:3]):
    print(f"{i+1:02d}. {c}")

## 13. Generate data statistics

In [ ]:
# Generate data statistics - sử dụng stats đã tính từ cell 8 và 9
print("="*80)
print("📊 DATA STATISTICS (từ cell 8 và 9)")
print("="*80)
print(f"✅ Mel Statistics:")
print(f"   Mean: {CALCULATED_MEAN:.4f}")
print(f"   Std:  {CALCULATED_STD:.4f}")
print(f"\n✅ Pitch Statistics:")
print(f"   Mean: {PITCH_MEAN:.4f}")
print(f"   Std:  {PITCH_STD:.4f}")
print(f"\n✅ Energy Statistics:")
print(f"   Mean: {ENERGY_MEAN:.4f}")
print(f"   Std:  {ENERGY_STD:.4f}")
print("="*80)
print("\n📝 Các stats này sẽ được dùng trong training script (cell 9)")


## 14. Kiểm tra symbols trong filelist

In [ ]:
# Kiểm tra symbols trong filelist
from matcha.text import symbols

bad_lines = []
print(f"🔍 Đang kiểm tra symbols trong filelist: {TRAIN_LIST_OUTPUT}")
print(f"📊 Tổng số symbols có sẵn: {len(symbols)}")

with open(TRAIN_LIST_OUTPUT, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            parts = line.strip().split("|")
            if len(parts) < 3:
                print(f"❌ Dòng {i}: format sai (cần 3 cột), lỗi: {line[:50]}")
                continue
            path, raw_text, ipa_text = parts[0], parts[1], parts[2]
        except ValueError as e:
            print(f"❌ Dòng {i}: lỗi parse: {e}")
            continue

        # Kiểm tra IPA text (cột 3)
        for c in ipa_text:
            if c not in symbols:
                bad_lines.append((i, c, ipa_text))
                break

if bad_lines:
    print(f"\n⚠️ Tìm thấy {len(bad_lines)} dòng có ký tự không hợp lệ:")
    for i, c, text in bad_lines[:10]:
        print(f"   Dòng {i}: ký tự '{c}' (ord={ord(c)}) → {text[:60]}")
    if len(bad_lines) > 10:
        print(f"   ... và {len(bad_lines) - 10} dòng khác")
else:
    print(f"\n✅ Tất cả ký tự trong filelist đều hợp lệ!")
    
print(f"\n✅ Hoàn tất kiểm tra {i} dòng")


## 16. Xem TensorBoard

In [ ]:
%load_ext tensorboard

# ✅ FIX: Update path to match training script output
# Training script saves logs to: /kaggle/working/IPIATTS/outputs/logs/
%tensorboard --logdir /kaggle/working/IPIATTS/outputs/logs --port 6006

## 17. Phân tích training loss

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorboard.backend.event_processing import event_accumulator
import os

# ✅ FIX: Update đường dẫn TensorBoard logs cho đúng với training script
# Training script lưu ở: outputs/logs/
log_dir = "/kaggle/working/IPIATTS/outputs/logs"

# Tìm version folder (version_0, version_1, etc.)
if os.path.exists(log_dir):
    versions = [d for d in os.listdir(log_dir) if d.startswith("version_")]
    if versions:
        # Lấy version mới nhất
        latest_version = sorted(versions)[-1]
        tensorboard_path = os.path.join(log_dir, latest_version)
        print(f"📂 Sử dụng logs từ: {tensorboard_path}")
    else:
        print("⚠️ Không tìm thấy version folder trong logs!")
        tensorboard_path = log_dir
else:
    print(f"❌ Không tìm thấy log directory: {log_dir}")
    print("💡 Đảm bảo đã chạy training (cell 10) trước khi xem logs!")
    tensorboard_path = None

if tensorboard_path and os.path.exists(tensorboard_path):
    try:
        ea = event_accumulator.EventAccumulator(tensorboard_path)
        ea.Reload()

        # Lấy dữ liệu loss
        scalars = ea.Scalars('loss/train_step')
        steps = [s.step for s in scalars]
        values = [s.value for s in scalars]

        # Vẽ đồ thị
        plt.figure(figsize=(10,5))
        plt.plot(steps, values, label='Train Loss', linewidth=1.5)
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.title('Biểu đồ Loss theo Step (Matcha-TTS)')
        plt.legend()
        plt.grid(True)
        plt.show()

        print(f"✅ Loss cuối cùng: {scalars[-1].value:.4f}")
        print(f"📊 Trung bình 10 step cuối: {np.mean([s.value for s in scalars[-10:]]):.4f}")
    except Exception as e:
        print(f"❌ Lỗi khi đọc TensorBoard logs: {e}")
        print("💡 Có thể training chưa chạy hoặc chưa có đủ dữ liệu")
else:
    print("⏭️ Skip cell này - chạy lại sau khi training xong!")

## 20. Utilities và Helper Functions

In [ ]:
# Kiểm tra checkpoint
def check_checkpoint(ckpt_path):
    import torch
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print("Checkpoint keys:", ckpt.keys())
    if 'epoch' in ckpt:
        print(f"Epoch: {ckpt['epoch']}")
    if 'global_step' in ckpt:
        print(f"Global step: {ckpt['global_step']}")
    return ckpt

# Tính toán mel statistics
def calculate_mel_stats(filelist_path, audio_root):
    import librosa
    import numpy as np
    from tqdm import tqdm
    
    mels = []
    with open(filelist_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f.readlines()[:100]):  # Sample 100 files
            audio_path = line.strip().split('|')[0]
            y, sr = librosa.load(audio_path, sr=22050)
            mel = librosa.feature.melspectrogram(
                y=y, sr=sr, n_fft=1024, hop_length=256, 
                win_length=1024, n_mels=80, fmin=0, fmax=8000
            )
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mels.append(mel_db)
    
    mels = np.concatenate(mels, axis=1)
    mel_mean = np.mean(mels)
    mel_std = np.std(mels)
    
    print(f"Mel Mean: {mel_mean}")
    print(f"Mel Std: {mel_std}")
    return mel_mean, mel_std

## 21. Find và list checkpoints

In [ ]:
# ✅ Tìm tất cả checkpoints (cross-platform)
import os
from pathlib import Path

# Xác định các thư mục gốc có thể chứa outputs
candidate_roots = []
# Kaggle working dir
for base in [
    "/kaggle/working/IPIATTS/outputs",
    "/kaggle/working/outputs",
    "/kaggle/working/IPIATTS/IPIATTS/outputs",
    str(Path.cwd() / "outputs"),
]:
    if os.path.isdir(base):
        candidate_roots.append(os.path.abspath(base))

# Loại trùng lặp, giữ thứ tự đầu tiên
seen = set()
search_roots = [x for x in candidate_roots if not (x in seen or seen.add(x))]

found_ckpts = []
for root in search_roots:
    for dirpath, _, filenames in os.walk(root):
        for name in filenames:
            if name.endswith(".ckpt"):
                full = os.path.join(dirpath, name)
                try:
                    mtime = os.path.getmtime(full)
                except Exception:
                    mtime = 0
                found_ckpts.append((mtime, full))

# Sắp xếp mới nhất trước và in tối đa 20
found_ckpts.sort(reverse=True)
found_ckpts = [p for _, p in found_ckpts]

print("🔎 Search roots:")
for r in search_roots:
    print("  ", r)

if found_ckpts:
    print("\n✅ Found checkpoints (top 20):")
    for p in found_ckpts[:20]:
        print(p)
else:
    print("\n❌ Không tìm thấy checkpoint trong các đường dẫn trên.")


## 22. Test inference sau khi train

In [ ]:
# Load model từ checkpoint (robust)
from matcha.models.matcha_tts import MatchaTTS
import torch
import torch.serialization
import torch.optim
import torch.nn
import argparse
import os
from pathlib import Path

# ✅ Allowlist các class cần thiết cho PyTorch 2.6+ (unpickle)
torch.serialization.add_safe_globals([
    argparse.Namespace,
    torch.optim.Adam,
    torch.optim.AdamW,  # optimizer AdamW trong checkpoint
])

# Ưu tiên dùng danh sách tìm được từ cell trước
checkpoint_path = None
if 'found_ckpts' in globals() and found_ckpts:
    checkpoint_path = found_ckpts[0]

# Nếu chưa có, thử các đường dẫn hay dùng (Kaggle + local)
if checkpoint_path is None:
    candidate_checkpoints = [
        "/kaggle/working/IPIATTS/outputs/matcha_prosody/checkpoints/last.ckpt",
        "/kaggle/working/IPIATTS/outputs/matcha_prosody_fixed/checkpoints/last.ckpt",
        "/kaggle/working/IPIATTS/outputs/matcha_prosody/last.ckpt",
        "/kaggle/working/IPIATTS/IPIATTS/outputs/matcha_prosody/checkpoints/last.ckpt",
        str(Path.cwd() / "outputs/matcha_prosody/checkpoints/last.ckpt"),
        str(Path.cwd() / "outputs/matcha_prosody/last.ckpt"),
    ]
    checkpoint_path = next((p for p in candidate_checkpoints if os.path.exists(p)), None)

if checkpoint_path is None:
    print("❌ Không tìm thấy checkpoint. Hãy chạy cell 'Find checkpoints' và kiểm tra lại.")
    model = None
else:
    print(f"📥 Loading checkpoint: {checkpoint_path}")
    try:
        model = MatchaTTS.load_from_checkpoint(checkpoint_path)
        model.eval()
        if torch.cuda.is_available():
            model = model.cuda()
        print("✅ Model loaded successfully!")
        print(f"📁 Checkpoint: {checkpoint_path}")
    except Exception as e:
        print("❌ Lỗi khi load checkpoint lần 1:", e)
        # Một số checkpoint cần weights_only=False (chỉ làm khi bạn tin cậy checkpoint này)
        try:
            print("🔁 Thử lại với weights_only=False...")
            model = MatchaTTS.load_from_checkpoint(checkpoint_path, weights_only=False)
            model.eval()
            if torch.cuda.is_available():
                model = model.cuda()
            print("✅ Model loaded successfully with weights_only=False!")
            print(f"📁 Checkpoint: {checkpoint_path}")
        except Exception as e2:
            print("❌ Vẫn lỗi khi load checkpoint:", e2)
            model = None


In [ ]:
# Synthesize speech với HiFi-GAN vocoder
from matcha.text import text_to_sequence
from matcha.utils.utils import intersperse
import soundfile as sf
import sys
import os
import torch
from argparse import Namespace
import numpy as np

# Đảm bảo model đã được load
if "model" not in globals() or model is None:
    raise RuntimeError("Model chưa được load. Chạy cell Load model và đảm bảo checkpoint tồn tại.")

# Add HiFi-GAN to path
sys.path.append('/kaggle/working/IPIATTS/matcha/hifigan')
from matcha.hifigan.models import Generator
from matcha.hifigan.config import v1

# ===== BƯỚC 1: TẠO MEL-SPECTROGRAM TỪ TEXT =====
text = "xin chào các bạn"
print(f"📝 Text input: {text}")

x = torch.tensor(
    intersperse(text_to_sequence(text, ["basic_cleaners_phothong"])[0], 0),
    dtype=torch.long
)[None]

# ✅ FIX: Add x_lengths parameter (required by synthesise method)
x_lengths = torch.LongTensor([x.shape[-1]])

if torch.cuda.is_available():
    x = x.cuda()
    x_lengths = x_lengths.cuda()

print("🔊 Generating mel-spectrogram...")
with torch.no_grad():
    output = model.synthesise(x, x_lengths, n_timesteps=10)
    mel = output['mel']

# Diagnostics on mel
mel_np = mel.detach().cpu().numpy()
print(f"   mel shape={mel_np.shape}, min={mel_np.min():.4f}, max={mel_np.max():.4f}, mean={mel_np.mean():.4f}")
if not np.isfinite(mel_np).all():
    raise ValueError("Mel contains NaN/Inf; check upstream processing.")

# ===== BƯỚC 2: LOAD HIFI-GAN VOCODER =====
print("\n📦 Loading HiFi-GAN vocoder...")
hifigan_checkpoint = "/kaggle/working/IPIATTS/matcha/hifigan/checkpoints/g_02500000"
print(f"   HiFi-GAN checkpoint: {hifigan_checkpoint}")
if not os.path.exists(hifigan_checkpoint):
    raise FileNotFoundError(f"Không tìm thấy HiFi-GAN checkpoint: {hifigan_checkpoint}")

# ✅ FIX: Convert dict to Namespace so Generator can access attributes
hifigan_config = Namespace(**v1) if isinstance(v1, dict) else v1

# Create HiFi-GAN generator
vocoder = Generator(hifigan_config)
state_dict_g = torch.load(hifigan_checkpoint, map_location='cpu')
vocoder.load_state_dict(state_dict_g['generator'])
vocoder.eval()
vocoder.remove_weight_norm()

if torch.cuda.is_available():
    vocoder = vocoder.cuda()

print("✅ HiFi-GAN loaded!")

# ===== BƯỚC 3: CONVERT MEL → AUDIO =====
print("\n🎵 Converting mel to audio...")
with torch.no_grad():
    # ✅ Try to get data_statistics from model buffers/hparams; otherwise use training globals
    mel_denorm = mel
    if hasattr(model, "mel_mean") and hasattr(model, "mel_std"):
        mel_denorm = mel * model.mel_std + model.mel_mean
        print("   Using model buffers for mel stats")
    else:
        try:
            mel_mean = model.hparams.data_statistics['mel_mean']
            mel_std = model.hparams.data_statistics['mel_std']
            mel_denorm = mel * mel_std + mel_mean
            print(f"   Using model hparams stats: mean={mel_mean:.4f}, std={mel_std:.4f}")
        except Exception:
            try:
                mel_denorm = mel * CALCULATED_STD + CALCULATED_MEAN
                print(f"   Using training mel stats: mean={CALCULATED_MEAN:.4f}, std={CALCULATED_STD:.4f}")
            except Exception:
                print("   ⚠️ No mel stats found - using mel as-is")
                mel_denorm = mel

    mel_denorm_np = mel_denorm.detach().cpu().numpy()
    print(f"   mel_denorm stats: min={mel_denorm_np.min():.4f}, max={mel_denorm_np.max():.4f}, mean={mel_denorm_np.mean():.4f}")
    if not np.isfinite(mel_denorm_np).all():
        raise ValueError("Denorm mel contains NaN/Inf.")

    audio = vocoder(mel_denorm).squeeze()

    if torch.cuda.is_available():
        audio = audio.cpu()

    audio = audio.numpy()

print(f"✅ Generated audio shape: {audio.shape}")
print(f"   Duration: {len(audio) / 22050:.2f} seconds")

# ===== BƯỚC 4: SAVE AUDIO FILE =====
output_path = "/kaggle/working/synthesized_audio.wav"
sf.write(output_path, audio, 22050)
print(f"\n💾 Saved audio to: {output_path}")

# ===== BƯỚC 5: PLAY AUDIO IN NOTEBOOK =====
from IPython.display import Audio, display
print("\n🔊 Playing audio:")
display(Audio(audio, rate=22050))
